# emutrust: see it working in 5 minutes

This notebook walks through the whole idea of [emutrust](https://github.com/mqurban/emutrust) with pictures:

1. A **slow model** (here a cheap stand-in so it runs fast)
2. **Run it** a few hundred times
3. **Train an AI emulator** that imitates it, and estimates its own uncertainty
4. **Check** the emulator on cases it has never seen
5. **Infer parameters** from noisy data, and see why including the emulator's error matters

Run each cell in order with **Shift + Enter**. The default CPU runtime is fine; no GPU is needed.

In [ ]:
!pip install -q emutrust emcee

In [ ]:
import time

import emcee
import matplotlib.pyplot as plt
import numpy as np

import emutrust
from emutrust import Emulator, GaussianLikelihood, ParameterSpace, run_model, validate
from emutrust.toy import TOY_K, TOY_SPACE, toy_spectrum

print("emutrust version", emutrust.__version__)

## 1. The "slow model"

`toy_spectrum` stands in for a slow physics code such as CAMB. It takes **3 parameters** (`amplitude`, `slope`, `cutoff`) and returns a **curve of 50 numbers**, shaped like a matter power spectrum. A real model might take seconds per call; this one is instant, so the notebook runs quickly.

Below: the curve for a few different parameter values. Notice it spans several orders of magnitude, so the y-axis is logarithmic.

In [ ]:
space = ParameterSpace(TOY_SPACE)
print(space)

fig, ax = plt.subplots(figsize=(7, 4))
for params in [
    {"amplitude": 1.0, "slope": -1.0, "cutoff": 3.0},
    {"amplitude": 2.0, "slope": -1.0, "cutoff": 3.0},
    {"amplitude": 1.0, "slope": -1.8, "cutoff": 3.0},
    {"amplitude": 1.0, "slope": -1.0, "cutoff": 1.2},
]:
    ax.loglog(TOY_K, toy_spectrum(params), label=", ".join(f"{k}={v}" for k, v in params.items()))
ax.set_xlabel("scale k")
ax.set_ylabel("model output")
ax.set_title("The model's output for different parameters")
ax.legend(fontsize=7)
plt.show()

## 2. Run the model at many parameter values

The emulator learns from examples. We choose 200 training points and 150 separate validation points using **Latin hypercube sampling**, which spreads them evenly so every part of each parameter's range is covered.

In [ ]:
X_train = space.sample(200, seed=0)
X_valid = space.sample(150, seed=1)

training = run_model(toy_spectrum, X_train, space, progress=False)
validation = run_model(toy_spectrum, X_valid, space, progress=False)
print("training runs:", training.X.shape, "->", training.Y.shape)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X_train[:, 0], X_train[:, 1], s=10)
ax.set_xlabel("amplitude")
ax.set_ylabel("slope")
ax.set_title("Training points: evenly spread (Latin hypercube)")
plt.show()

## 3. Train the emulator

`Emulator.fit` trains a small **ensemble** of neural networks. Each starts from different random weights, so where they agree we can trust the prediction, and where they disagree the emulator is uncertain. This takes about 10 to 30 seconds.\n#\n# In the bottom row, the red line is the emulator's actual error and the shaded band is how large it *thinks* its error is. If the uncertainty is honest, the line should mostly stay inside the band.

Notice also that the error moves **smoothly** across scales: neighbouring points share the same mistake. Errors like this do not average out over many data points, which is why emutrust measures how the errors move together.

In [ ]:
t0 = time.time()
emu = Emulator.fit(training, n_members=5, hidden=(64, 64), epochs=600, seed=0)
print(f"trained in {time.time() - t0:.0f} s")

# Compare with the true model at 3 points it has never seen.
# Top: the curves (they overlap almost perfectly).
# Bottom: the emulator's error in %, with the emulator's own +-2 std uncertainty band.
fig, axes = plt.subplots(2, 3, figsize=(14, 6.5), sharex=True)
for col, i in enumerate([0, 1, 2]):
    mean, std = emu.predict(validation.X[i])
    true = validation.Y[i]
    top, bottom = axes[0, col], axes[1, col]
    top.loglog(TOY_K, true, "k-", lw=2, label="true model")
    top.loglog(TOY_K, mean, "r--", label="emulator")
    top.set_title(
        ", ".join(f"{k}={v:.2f}" for k, v in space.to_dict(validation.X[i]).items()), fontsize=8
    )
    bottom.semilogx(TOY_K, 100 * (mean - true) / true, "r-", label="emulator error")
    bottom.fill_between(
        TOY_K,
        -200 * std / true,
        200 * std / true,
        color="r",
        alpha=0.2,
        label="emulator's own ± 2 std",
    )
    bottom.axhline(0, color="k", lw=0.8)
    bottom.set_xlabel("scale k")
axes[0, 0].legend(fontsize=8)
axes[1, 0].legend(fontsize=8)
axes[1, 0].set_ylabel("error (%)")
plt.suptitle("Emulator vs the true model, on cases it never saw")
plt.tight_layout()
plt.show()

## 4. Check the emulator on unseen runs

`validate` compares the emulator with the 150 validation runs and reports:

- **Accuracy**: how big its errors are, and where they are largest
- **Uncertainty calibration**: whether its own error estimates are honest (the truth should fall within 1 predicted std about 68% of the time)

The left plot should look like the dashed bell curve if the uncertainty is honest.

In [ ]:
report = validate(emu, validation)
print(report)
report.plot()
plt.show()

## 5. Infer parameters from noisy data

Now the real use: we have a **noisy observation** and want to find the parameters that produced it, with error bars.

We pretend the truth is `amplitude=1.2, slope=-1.1, cutoff=3.0`, add 1% noise, and run MCMC (a guess-and-check search) twice:

- **Emulator treated as exact** (what people usually do)
- **With emutrust**: the emulator's measured error is included

Each run takes about 20 to 60 seconds.

In [ ]:
truth = np.array([1.2, -1.1, 3.0])
model = toy_spectrum(space.to_dict(truth))
rng = np.random.default_rng(3)
data = model * (1 + 0.01 * rng.standard_normal(model.size))

chains = {}
for label, error in [("emulator treated as exact", None), ("with emutrust", report)]:
    like = GaussianLikelihood(emu, data, noise_std=0.01 * model, emulator_error=error)
    start = truth + 0.02 * (space.high - space.low) * rng.standard_normal((32, space.dim))
    sampler = emcee.EnsembleSampler(32, space.dim, like, vectorize=True)
    sampler.run_mcmc(start, 3000, progress=False)
    chains[label] = sampler.get_chain(discard=1000, flat=True)
    print(f"done: {label}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for j, (ax, name) in enumerate(zip(axes, space.names)):
    for label, color in [("emulator treated as exact", "#c0392b"), ("with emutrust", "#1f4e79")]:
        ax.hist(
            chains[label][:, j],
            bins=60,
            density=True,
            histtype="step",
            lw=2,
            color=color,
            label=label,
        )
    ax.axvline(truth[j], color="k", ls="--", label="true value")
    ax.set_xlabel(name)
axes[0].legend(fontsize=8)
plt.suptitle("What each method believes about the parameters")
plt.tight_layout()
plt.show()

print("parameter     true    exact: 68% interval        emutrust: 68% interval")
for j, name in enumerate(space.names):
    row = f"{name:>9}  {truth[j]:6.3f}"
    for label in chains:
        lo, hi = np.percentile(chains[label][:, j], [15.87, 84.13])
        inside = "yes" if lo <= truth[j] <= hi else "no "
        row += f"    [{lo:7.3f}, {hi:7.3f}] contains truth: {inside}"
    print(row)

**How to read this:** the red curves (emulator treated as exact) are usually **narrower**: they claim more certainty. When the emulator's small errors push the answer slightly off, narrow curves can miss the true value (dashed line). The blue curves (emutrust) are a little wider because they honestly include the emulator's error, so they contain the truth more reliably.

One run is a single example. The repository's [coverage tests](https://github.com/mqurban/emutrust#readme) repeat this over many simulated observations and count how often each method's 68% interval contains the truth: about 20 to 55% when the emulator is treated as exact, and close to 68% with emutrust.

## Try it yourself

- Change `truth` in the last cell and rerun it.
- Train with fewer runs (e.g. `space.sample(60, seed=0)` in step 2): the emulator gets worse, and the difference between red and blue grows.
- Replace `toy_spectrum` with **your own function** that takes a dict of parameters and returns an array.